# 可视化模块演示 (Viz Module)

本notebook演示hscredit库中可视化模块的全部功能，包含8种可视化方法。

In [ ]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")

In [ ]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
demo_feature = feature_cols[0]
print(f"演示特征: {demo_feature}")

## 1. 导入可视化模块

In [ ]:
from hscredit.core.viz import (
    bin_plot,
    corr_plot,
    ks_plot,
    hist_plot,
    psi_plot,
    dataframe_plot,
    distribution_plot,
    plot_weights
)

print("可视化模块导入成功！")

## 2. 分箱可视化 (bin_plot)

展示特征分箱后的分布和坏账率。

In [ ]:
from hscredit.core.binning import OptimalBinning

# 先进行分箱
binner = OptimalBinning(method='quantile', max_n_bins=5)
binner.fit(df[[demo_feature]], df[target_col])
bin_table = binner.get_bin_table(demo_feature)

# 分箱可视化
fig, ax = plt.subplots(figsize=(12, 6))
bin_plot(bin_table, ax=ax)
plt.title(f'{demo_feature} 分箱分析')
plt.tight_layout()
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/viz_bin_plot.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图片已保存至: {output_path}")

## 3. 相关性热力图 (corr_plot)

展示特征之间的相关性矩阵。

In [ ]:
# 选择部分特征
features_subset = feature_cols[:10]
X_subset = df[features_subset]

# 相关性热力图
fig, ax = plt.subplots(figsize=(12, 10))
corr_plot(X_subset, ax=ax)
plt.title('特征相关性热力图')
plt.tight_layout()
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/viz_corr_plot.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图片已保存至: {output_path}")

## 4. KS曲线图 (ks_plot)

展示模型KS曲线和ROC曲线。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from hscredit.core.encoders import WOEEncoder

# 准备数据
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# WOE编码
woe = WOEEncoder()
X_train_woe = woe.fit_transform(X_train, y_train)
X_test_woe = woe.transform(X_test)

# 训练逻辑回归
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_woe, y_train)
y_pred_proba = lr.predict_proba(X_test_woe)[:, 1]

# KS曲线图
fig, ax = plt.subplots(figsize=(10, 6))
ks_plot(y_test, y_pred_proba, ax=ax)
plt.title('KS曲线与ROC曲线')
plt.tight_layout()
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/viz_ks_plot.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图片已保存至: {output_path}")

## 5. 特征分布图 (hist_plot)

展示特征在好坏事样本上的分布对比。

In [ ]:
# 特征分布图
fig, ax = plt.subplots(figsize=(12, 6))
hist_plot(df[[demo_feature]], df[target_col], ax=ax, bins=30)
plt.title(f'{demo_feature} 分布对比')
plt.tight_layout()
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/viz_hist_plot.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图片已保存至: {output_path}")

## 6. PSI稳定性图 (psi_plot)

展示特征在训练集和测试集间的PSI稳定性。

In [ ]:
# PSI稳定性图
fig, ax = plt.subplots(figsize=(12, 6))
psi_plot(X_train[[demo_feature]], X_test[[demo_feature]], ax=ax)
plt.title(f'{demo_feature} PSI稳定性分析')
plt.tight_layout()
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/viz_psi_plot.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图片已保存至: {output_path}")

## 7. 分布时间图 (distribution_plot)

按时间维度展示特征分布变化。

In [ ]:
# 分布时间图
# 使用loan_date作为时间维度
df_time = df.copy()
df_time['year_month'] = pd.to_datetime(df_time['loan_date']).dt.to_period('M')

# 按月份聚合
monthly_stats = df_time.groupby('year_month')[demo_feature].agg(['mean', 'std', 'count']).reset_index()
monthly_stats.columns = ['时间', '均值', '标准差', '样本数']

print("月度分布统计:")
print(monthly_stats.head(10))

## 8. 逻辑回归系数图 (plot_weights)

展示逻辑回归模型的特征系数和置信区间。

In [ ]:
# 逻辑回归系数图
fig, ax = plt.subplots(figsize=(12, 8))
plot_weights(lr.coef_[0], feature_names=X_train_woe.columns.tolist(), ax=ax)
plt.title('逻辑回归特征系数')
plt.tight_layout()
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/viz_weights_plot.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图片已保存至: {output_path}")

## 9. 可视化结果汇总

将所有可视化结果汇总。

In [ ]:
# 可视化结果汇总
viz_files = [
    'viz_bin_plot.png',
    'viz_corr_plot.png',
    'viz_ks_plot.png',
    'viz_hist_plot.png',
    'viz_psi_plot.png',
    'viz_weights_plot.png'
]

output_dir = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples'
print("生成的可视化文件:")
for f in viz_files:
    filepath = os.path.join(output_dir, f)
    if os.path.exists(filepath):
        print(f"  ✓ {f}")
    else:
        print(f"  ✗ {f} (未生成)")